# Réseau de neurones simple avec PyTorch Lightning

Dans ces travaux pratiques, nous construisons avec PyTorch Lightning des réseaux de neurones qui reconnaissent les chiffres manuscrits de MNIST. Lightning écrit la boucle d'entraînement à notre place : le modèle reste un `nn.Module` PyTorch, et un `Trainer` l'entraîne (slides « PyTorch Lightning », « Définir un `LightningModule` » et « Entraîner avec le `Trainer` »).

1. une régression logistique multinomiale, le réseau sans couche cachée, avec deux façons de donner les classes à la perte, puis l'accord entre sa sortie et sa perte ;
2. un perceptron multicouche régularisé ;
3. le suivi de l'entraînement avec TensorBoard, pour comparer deux tailles de batch ;
4. une recherche d'hyper-paramètres avec Optuna.

Il n'est pas nécessaire d'utiliser une carte graphique pour ce TP : chaque entraînement est rapide sur CPU, et seule la recherche d'hyper-paramètres, la dernière partie, prend quelques minutes. Vous pouvez donc changer d'environnement si celui qui vous a été attribué a une carte graphique (`Exécution > Modifier le type d'exécution`).

In [ ]:
!pip install -q lightning torchmetrics optuna tensorboard
import pathlib

import lightning
import matplotlib.pyplot as plt
import optuna
import pandas
import torch
import torchmetrics
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
lightning.seed_everything(0)

## Récupération et préparation des données

Comme dans le TP précédent, torchvision fournit le jeu MNIST : 60 000 images d'entraînement et 10 000 de test, des chiffres manuscrits de 28 × 28 pixels en niveaux de gris, avec leur label, de 0 à 9. La cellule suivante les charge et les prépare de la même façon : chaque image aplatie en un vecteur de 784 pixels, l'entrée d'une couche `nn.Linear`, et ses pixels ramenés de $[0, 255]$ à $[0, 1]$.

In [ ]:
nb_classes = 10
dim_entree = 28 * 28

donnees_entrainement = torchvision.datasets.MNIST("donnees", train=True, download=True)
donnees_test = torchvision.datasets.MNIST("donnees", train=False, download=True)
# Des images « à plat », de dim_entree pixels dans [0, 1] : -1 laisse PyTorch
# calculer le nombre d'exemples
X_train = donnees_entrainement.data.reshape(-1, dim_entree) / 255
X_test = donnees_test.data.reshape(-1, dim_entree) / 255
y_train, y_test = donnees_entrainement.targets, donnees_test.targets
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

## Encodage *one-hot* des classes

Les labels `y_train` et `y_test` sont des indices de classe, des entiers de 0 à 9. Encodez-les en [*one-hot*](https://fr.wikipedia.org/wiki/Encodage_one-hot) : pour chaque exemple, 10 valeurs, toutes nulles sauf un 1 à l'indice de sa classe. C'est la première des deux façons de donner les classes à l'entropie croisée que nous verrons.

Rangez les classes encodées dans `Y_train` et `Y_test`, et gardez `y_train` et `y_test` : la suite se sert des deux.

**Documentation utile**

- [`torch.nn.functional.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.functional.one_hot(y_train, nb_classes)` renvoie un tenseur d'entiers de forme `(60000, 10)` ; `.float()` le convertit en flottants, le type qu'attend l'entropie croisée pour des cibles one-hot. Le label 5, celui du premier exemple, y devient `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
Y_train = nn.functional.one_hot(..., nb_classes).float()
Y_test = ...
```

### Solution

In [ ]:
# Les classes en vecteurs one-hot, en flottants pour l'entropie croisée
Y_train = nn.functional.one_hot(y_train, nb_classes).float()
Y_test = nn.functional.one_hot(y_test, nb_classes).float()
print(Y_train.shape, Y_test.shape)
print(f"Label {int(y_train[0])} en one-hot : {Y_train[0].tolist()}")

## Le `LightningModule` du TP

La cellule suivante définit les outils des parties suivantes :

- `ClassifieurMNIST`, le `LightningModule` du TP (slide « Définir un `LightningModule` ») : il enveloppe un modèle PyTorch qui renvoie des *logits*, les scores bruts des 10 classes, et se charge de la perte, de l'exactitude (*accuracy*, avec torchmetrics) et de l'optimiseur. La perte est l'entropie croisée, qui applique elle-même la softmax (slide « Piège classique : softmax et entropie croisée ») et accepte comme cibles des vecteurs one-hot aussi bien que des indices de classe. L'optimiseur, son learning rate et sa *weight decay* sont des paramètres ;
- `fabrique_chargeurs`, qui sépare des données en entraînement et validation (20 % par défaut) et renvoie leurs `DataLoader` ;
- `evalue`, qui affiche la perte et l'exactitude d'un modèle entraîné sur l'ensemble de test ;
- `trace_pertes`, qui trace, epoch par epoch, les pertes d'entraînement et de validation d'un entraînement, depuis le fichier `metrics.csv` que son `CSVLogger` écrit dans `trainer.logger.log_dir`.

In [ ]:
class ClassifieurMNIST(lightning.LightningModule):
  """Enveloppe Lightning : entropie croisée sur les logits, exactitude comme métrique."""

  def __init__(
    self,
    modele: nn.Module,
    optimiseur: type[torch.optim.Optimizer] = torch.optim.SGD,
    taux_apprentissage: float = 0.01,
    decroissance_poids: float = 0.0,
    nb_classes_sortie: int = nb_classes,
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["modele"])
    self.modele = modele
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, dim_entree)
    # torchmetrics demande une instance de métrique par étape
    self.exactitudes = nn.ModuleDict(
      {
        f"exactitude_{etape}": torchmetrics.Accuracy(
          task="multiclass", num_classes=nb_classes_sortie
        )
        for etape in ("entrainement", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.modele(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], etape: str) -> torch.Tensor:
    images, cibles = batch
    logits = self(images)
    perte = nn.functional.cross_entropy(logits, cibles)
    # La métrique, elle, a besoin d'indices de classe
    etiquettes = cibles.argmax(dim=-1) if cibles.dim() > 1 else cibles
    exactitude = self.exactitudes[f"exactitude_{etape}"]
    exactitude(logits, etiquettes)
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    self.log(
      f"exactitude_{etape}", exactitude, on_step=False, on_epoch=True, prog_bar=True
    )
    return perte

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "entrainement")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return self.hparams.optimiseur(
      self.parameters(),
      lr=self.hparams.taux_apprentissage,
      weight_decay=self.hparams.decroissance_poids,
    )


def fabrique_chargeurs(
  X: torch.Tensor,
  Y: torch.Tensor,
  taille_lot: int = 128,
  fraction_validation: float = 0.2,
) -> tuple[DataLoader, DataLoader]:
  """Sépare les données entre entraînement et validation."""
  dataset_entrainement, dataset_val = random_split(
    TensorDataset(X, Y), [1 - fraction_validation, fraction_validation]
  )
  return (
    DataLoader(dataset_entrainement, batch_size=taille_lot, shuffle=True),
    DataLoader(dataset_val, batch_size=taille_lot),
  )


def evalue(
  trainer: lightning.Trainer, classifieur: ClassifieurMNIST, one_hot: bool = False
) -> None:
  """Affiche la perte et l'exactitude sur l'ensemble de test.

  one_hot : True pour un modèle entraîné sur les cibles one-hot (Y_train).
  """
  chargeur_test = DataLoader(
    TensorDataset(X_test, Y_test if one_hot else y_test), batch_size=128
  )
  score = trainer.test(classifieur, chargeur_test, verbose=False)[0]
  print(
    f"Test : perte {score['perte_test']:.3f}, exactitude {score['exactitude_test']:.3f}"
  )


def trace_pertes(trainer: lightning.Trainer) -> None:
  """Trace les pertes d'entraînement et de validation qu'a écrites un CSVLogger."""
  # Les métriques d'entraînement et de validation ne sont pas écrites sur les
  # mêmes lignes du fichier CSV : on les regroupe par epoch
  mesures = (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["perte_entrainement", "perte_val"])
  )
  plt.plot(mesures.index, mesures["perte_entrainement"], label="Entraînement")
  plt.plot(mesures.index, mesures["perte_val"], label="Validation")
  plt.xlabel("Epoch")
  plt.ylabel("Perte")
  plt.legend()
  plt.show()

## Régression logistique multinomiale

Créez `modele`, un réseau sans couche cachée : chacune de ses `nb_classes` sorties est une combinaison linéaire des `dim_entree` pixels, un logit, et la softmax de l'entropie croisée en fait 10 probabilités de somme 1. C'est une régression logistique multinomiale.

Enveloppez-le dans `classifieur`, un `ClassifieurMNIST` avec ses réglages par défaut. Affichez son résumé avec `ModelSummary`, et expliquez son nombre de paramètres.

Entraînez ensuite `classifieur` avec un `lightning.Trainer` (slide « Entraîner avec le `Trainer` ») :

- des batchs de 128 exemples, avec 20 % des données d'entraînement en validation : `fabrique_chargeurs` sur `X_train` et les cibles one-hot `Y_train` ;
- 10 epochs ;
- un `CSVLogger`, qui enregistre les métriques dans `journaux/simple`, et sans sauvegarde de checkpoints.

Tracez ses courbes de perte avec `trace_pertes` : la perte de validation descend-elle encore à la dernière epoch ? Mesurez enfin sa perte et son exactitude sur l'ensemble de test avec `evalue`. Gardez ces chiffres : ils servent de référence aux modèles suivants.

**Documentation utile**

- [`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [`CSVLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.CSVLogger.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Sequential(nn.Linear(dim_entree, nb_classes))`
- `ClassifieurMNIST(modele)` : SGD, avec un learning rate de 0,01
- `print(ModelSummary(classifieur, max_depth=-1))` : `max_depth=-1` détaille toutes les couches, y compris celles du modèle enveloppé
- `fabrique_chargeurs(X_train, Y_train, taille_lot=128)` renvoie les `DataLoader` d'entraînement et de validation
- `lightning.Trainer(max_epochs=10, accelerator="auto", devices=1, logger=CSVLogger("journaux", name="simple"), enable_checkpointing=False)` : `accelerator="auto"` prend le GPU s'il y en a un
- `callbacks=[TQDMProgressBar()]` : la barre de progression de tqdm, plutôt que celle de rich, que Lightning choisit quand rich est installé (Colab) et qui a figé des entraînements
- `trainer.fit(classifieur, chargeur_entrainement, chargeur_val)`, puis `trace_pertes(trainer)`
- `evalue(trainer, classifieur, one_hot=True)` : le modèle apprend sur des cibles one-hot
- Résultat attendu : 7 850 paramètres ; environ 89 % d'exactitude en validation comme en test ; SGD, avec un learning rate de 0,01, progresse encore à la dixième epoch

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele = nn.Sequential(nn.Linear(..., ...))
classifieur = ClassifieurMNIST(...)
print(ModelSummary(..., max_depth=-1))

chargeur_entrainement, chargeur_val = fabrique_chargeurs(..., ..., taille_lot=...)
trainer = lightning.Trainer(
  max_epochs=...,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name=...),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(..., ..., ...)
trace_pertes(...)
evalue(trainer, classifieur, one_hot=...)
```

### Solution

In [ ]:
modele = nn.Sequential(nn.Linear(dim_entree, nb_classes))
classifieur = ClassifieurMNIST(modele)
print(ModelSummary(classifieur, max_depth=-1))

# Une couche linéaire a un poids par entrée et par sortie, plus un biais par
# sortie : 784 × 10 + 10 = 7 850 paramètres

In [ ]:
# Le Trainer exécute la boucle d'entraînement complète du LightningModule
chargeur_entrainement, chargeur_val = fabrique_chargeurs(
  X_train, Y_train, taille_lot=128
)

trainer = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name="simple"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
# La perte de validation descend encore à la dixième epoch : avec un learning
# rate de 0,01, SGD n'a pas fini de converger
trace_pertes(trainer)
evalue(trainer, classifieur, one_hot=True)

## Accorder la sortie et la perte

L'entropie croisée de PyTorch (`nn.functional.cross_entropy`, dans `ClassifieurMNIST`) accepte aussi, comme cibles, les labels entiers `y_train` (des indices de classe, de type `torch.int64`) : l'encodage one-hot devient inutile, et les modèles suivants s'en passent.

Elle attend toujours des logits, et applique elle-même la softmax. Un modèle qui se termine par une softmax lui donne donc des probabilités, qu'elle lit comme des logits : la mauvaise paire.

Entraînez sur `y_train`, avec l'optimiseur `torch.optim.Adam`, un learning rate de 10⁻³ et les réglages de la partie précédente, deux régressions logistiques :

1. `classifieur_adam`, qui enveloppe un nouveau modèle identique à `modele` (`CSVLogger` dans `journaux/adam`) ;
2. `classifieur_softmax`, dont le modèle se termine par une couche `nn.Softmax` (`CSVLogger` dans `journaux/softmax`).

Comparez leur perte et leur exactitude de test : laquelle a une perte fausse ? Apprend-elle moins bien ? Pourquoi PyTorch ne signale-t-il aucune erreur ?

**Documentation utile**

- [`torch.nn.Softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Sequential(nn.Linear(dim_entree, nb_classes), nn.Softmax(dim=-1))` : la softmax sur la dernière dimension, celle des classes
- `ClassifieurMNIST(modele, optimiseur=torch.optim.Adam, taux_apprentissage=1e-3)`
- `fabrique_chargeurs(X_train, y_train, taille_lot=128)`, un nouveau `Trainer` par modèle, puis `evalue(trainer, classifieur)`, sans `one_hot`
- Une fonction `entraine(modele, nom)` qui enveloppe, entraîne et évalue un modèle évite d'écrire deux fois le même code
- Résultat attendu : environ 92 % d'exactitude en test pour `classifieur_adam`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def entraine(modele: nn.Module, nom: str) -> ClassifieurMNIST:
  classifieur = ClassifieurMNIST(modele, optimiseur=..., taux_apprentissage=...)
  chargeur_entrainement, chargeur_val = fabrique_chargeurs(X_train, ..., taille_lot=128)
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("journaux", name=nom),
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
  evalue(trainer, classifieur)
  return classifieur


classifieur_adam = entraine(nn.Sequential(nn.Linear(dim_entree, nb_classes)), "adam")
classifieur_softmax = entraine(nn.Sequential(..., ...), "softmax")
```

### Solution

In [ ]:
def entraine(modele: nn.Module, nom: str) -> ClassifieurMNIST:
  """Entraîne et évalue un modèle avec Adam, sur les labels entiers y_train."""
  classifieur = ClassifieurMNIST(
    modele, optimiseur=torch.optim.Adam, taux_apprentissage=1e-3
  )
  chargeur_entrainement, chargeur_val = fabrique_chargeurs(
    X_train, y_train, taille_lot=128
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("journaux", name=nom),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
  evalue(trainer, classifieur)
  return classifieur


# Bonne paire : le modèle renvoie des logits, l'entropie croisée applique la softmax
classifieur_adam = entraine(nn.Sequential(nn.Linear(dim_entree, nb_classes)), "adam")
# Mauvaise paire : une softmax en trop, avant celle de l'entropie croisée
classifieur_softmax = entraine(
  nn.Sequential(nn.Linear(dim_entree, nb_classes), nn.Softmax(dim=-1)), "softmax"
)

# Des exactitudes proches, environ 92,5 % en test, 3 points de plus qu'avec SGD à
# architecture égale, mais une perte fausse pour classifieur_softmax : environ 1,55
# au lieu de 0,27. La softmax de l'entropie croisée s'applique à des probabilités,
# entre 0 et 1 : la probabilité qu'elle donne à la bonne classe ne dépasse jamais
# e / (e + 9), soit 23 %, et la perte ne descend pas sous 1,46. Les gradients en sont
# écrasés ; Adam, qui adapte la taille de ses pas, compense ici, mais avec SGD la
# même régression apprend bien plus lentement (environ 79 % contre 89 % après 10
# epochs). PyTorch ne voit pas d'erreur : des probabilités ont la forme et le type de
# logits.

## Perceptron multicouche régularisé

Écrivez une fonction `construit_modele_profond` qui renvoie un perceptron multicouche neuf :

- 4 couches cachées de 20 neurones, avec l'activation ReLU ;
- pour toutes les couches, l'initialisation de He, adaptée à ReLU (`nn.init.kaiming_normal_`, slide « Initialisation des poids ») ;
- une couche de sortie de `nb_classes` neurones, sans activation : elle renvoie des logits, comme la régression logistique.

Enveloppez-en un dans `classifieur_profond`, un `ClassifieurMNIST` qui l'entraîne avec l'optimiseur `torch.optim.AdamW`, un learning rate de 10⁻³ et une *weight decay* de 0,01 : la régularisation est portée par l'optimiseur, qui réduit les poids à chaque pas pour freiner le sur-apprentissage (slide « AdamW et régularisation des poids »). Affichez son résumé, puis entraînez-le sur les labels entiers avec les mêmes réglages que la régression logistique. Combien de points d'exactitude en test gagne-t-il sur `classifieur_adam`, la régression logistique entraînée avec Adam ? Pour combien de paramètres de plus ? Comparez aussi les exactitudes d'entraînement et de validation de la dernière epoch : ce réseau sur-apprend-il ?

Les parties suivantes réutilisent `construit_modele_profond`.

**Documentation utile**

- [`torch.nn.init.kaiming_normal_`](https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_normal_)

In [ ]:
def construit_modele_profond() -> nn.Module: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.init.kaiming_normal_(couche.weight, nonlinearity="relu")` remplace sur place les poids d'une couche `nn.Linear` par des tirages d'une loi normale de variance 2 / n_entrée, qui gardent la variance du signal d'une couche ReLU à l'autre ; l'initialisation par défaut de PyTorch, uniforme, n'est pas faite pour ReLU
- Une fonction `cree_dense(nb_entrees, nb_sorties)` qui crée la couche et initialise ses poids évite de répéter ces deux lignes
- `nn.Sequential(cree_dense(dim_entree, 20), nn.ReLU(), ...)` : chaque couche cachée est suivie de son activation
- `ClassifieurMNIST(construit_modele_profond(), optimiseur=torch.optim.AdamW, taux_apprentissage=1e-3, decroissance_poids=0.01)`
- `trainer_profond.callback_metrics["exactitude_entrainement"]` et `["exactitude_val"]` : les exactitudes d'entraînement et de validation de la dernière epoch, à lire avant `evalue`, dont le test les remplace
- Résultat attendu : 17 170 paramètres, environ 95,5 % d'exactitude en test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def cree_dense(nb_entrees: int, nb_sorties: int) -> nn.Linear:
  couche = nn.Linear(nb_entrees, nb_sorties)
  nn.init.kaiming_normal_(..., nonlinearity=...)
  return couche


def construit_modele_profond() -> nn.Module:
  return nn.Sequential(
    cree_dense(dim_entree, 20),
    nn.ReLU(),
    ...,
    cree_dense(20, nb_classes),
  )


classifieur_profond = ClassifieurMNIST(
  construit_modele_profond(),
  optimiseur=...,
  taux_apprentissage=...,
  decroissance_poids=...,
)
print(ModelSummary(classifieur_profond, max_depth=-1))
chargeur_entrainement, chargeur_val = fabrique_chargeurs(
  X_train, y_train, taille_lot=128
)
trainer_profond = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name="profond"),
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer_profond.fit(..., ..., ...)
print(trainer_profond.callback_metrics[...], trainer_profond.callback_metrics[...])
evalue(trainer_profond, classifieur_profond)
```

### Solution

In [ ]:
def cree_dense(nb_entrees: int, nb_sorties: int) -> nn.Linear:
  """Une couche linéaire aux poids initialisés par la méthode de He, pour ReLU."""
  couche = nn.Linear(nb_entrees, nb_sorties)
  nn.init.kaiming_normal_(couche.weight, nonlinearity="relu")
  return couche


def construit_modele_profond() -> nn.Module:
  """Un perceptron multicouche neuf : 4 couches cachées de 20 neurones."""
  return nn.Sequential(
    cree_dense(dim_entree, 20),
    nn.ReLU(),
    cree_dense(20, 20),
    nn.ReLU(),
    cree_dense(20, 20),
    nn.ReLU(),
    cree_dense(20, 20),
    nn.ReLU(),
    cree_dense(20, nb_classes),
  )


# La régularisation est portée par l'optimiseur : AdamW applique la weight decay
# directement aux poids, alors qu'une pénalité L2 ajoutée à la perte serait
# déformée par les pas adaptatifs d'Adam
classifieur_profond = ClassifieurMNIST(
  construit_modele_profond(),
  optimiseur=torch.optim.AdamW,
  taux_apprentissage=1e-3,
  decroissance_poids=0.01,
)
print(ModelSummary(classifieur_profond, max_depth=-1))

chargeur_entrainement, chargeur_val = fabrique_chargeurs(
  X_train, y_train, taille_lot=128
)
trainer_profond = lightning.Trainer(
  max_epochs=10,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("journaux", name="profond"),
  # La barre de progression de tqdm, plutôt que celle de rich que Lightning
  # choisit quand rich est installé (Colab) : elle a figé des entraînements
  callbacks=[TQDMProgressBar()],
  enable_checkpointing=False,
)
trainer_profond.fit(classifieur_profond, chargeur_entrainement, chargeur_val)
# Les métriques de la dernière epoch, avant que le test ne les remplace
exactitude_entrainement = float(
  trainer_profond.callback_metrics["exactitude_entrainement"]
)
exactitude_val = float(trainer_profond.callback_metrics["exactitude_val"])
print(
  f"Exactitude de la dernière epoch : entraînement {exactitude_entrainement:.3f}, "
  f"validation {exactitude_val:.3f}"
)
evalue(trainer_profond, classifieur_profond)

# Environ 3 points de plus que classifieur_adam (95,5 % contre 92,5 %), pour 17 170
# paramètres au lieu de 7 850 : le gain de l'architecture. Contre la régression
# logistique entraînée avec SGD, les 6 points d'écart mêlent architecture et
# optimiseur. À la dernière epoch, l'exactitude d'entraînement dépasse celle de validation d'un à
# deux points : un début de sur-apprentissage, que la weight decay n'empêche pas à
# elle seule.

## Taille des batchs sous TensorBoard

TensorBoard affiche les courbes d'apprentissage qu'un `TensorBoardLogger`, passé au `Trainer` par son argument `logger`, écrit dans un dossier : un sous-dossier par entraînement. Lancez la cellule suivante avant l'exercice : le tableau de bord se remplit au fil des epochs (bouton de rechargement en haut à droite). Dans un Codespace, si le tableau de bord ne s'affiche pas sous la cellule, lancez `tensorboard --logdir journaux` dans un terminal, depuis le dossier du notebook, et ouvrez l'adresse qu'il affiche.

Entraînez ensuite deux perceptrons multicouches neufs (`construit_modele_profond()`) avec les réglages de `classifieur_profond` (`torch.optim.AdamW`, un learning rate de 10⁻³, une *weight decay* de 0,01, 10 epochs), l'un sur des batchs de 128 exemples, l'autre sur des batchs de 1 500. Passez à chaque `Trainer` un `TensorBoardLogger` qui écrit dans `journaux/lot_128` pour le premier, dans `journaux/lot_1500` pour le second.

Comparez les courbes `perte_val` et `exactitude_val` des deux entraînements : lequel converge le plus vite, epoch par epoch ? Combien de mises à jour des poids chacun fait-il par epoch ? Pourquoi le grand batch converge-t-il moins vite par epoch ?

**Documentation utile**

- [`TensorBoardLogger`](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.TensorBoardLogger.html)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir journaux

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `fabrique_chargeurs(X_train, y_train, taille_lot=taille_lot)` : de nouveaux `DataLoader` pour chaque taille de batch
- `TensorBoardLogger("journaux", name="lot_128")` : les courbes vont dans `journaux/lot_128`
- Un modèle neuf par entraînement, `construit_modele_profond()` : sinon le second entraînement repart des poids du premier
- Dans TensorBoard, l'onglet *Scalars* superpose les courbes des deux dossiers ; `CSVLogger` écrit aussi dans `journaux`, mais TensorBoard ignore ses fichiers
- Relancer un entraînement crée un nouveau sous-dossier `version_<n>` : supprimez `journaux` pour repartir d'un tableau de bord propre
- Pour l'expliquer, comptez les mises à jour des poids par epoch : les 48 000 exemples d'entraînement divisés par la taille des batchs

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for taille_lot in (128, 1500):
  chargeur_entrainement, chargeur_val = fabrique_chargeurs(
    X_train, y_train, taille_lot=...
  )
  classifieur = ClassifieurMNIST(
    construit_modele_profond(),
    optimiseur=...,
    taux_apprentissage=...,
    decroissance_poids=...,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("journaux", name=f"lot_{...}"),
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
```

### Solution

In [ ]:
for taille_lot in (128, 1500):
  chargeur_entrainement, chargeur_val = fabrique_chargeurs(
    X_train, y_train, taille_lot=taille_lot
  )
  classifieur = ClassifieurMNIST(
    construit_modele_profond(),
    optimiseur=torch.optim.AdamW,
    taux_apprentissage=1e-3,
    decroissance_poids=0.01,
  )
  trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=TensorBoardLogger("journaux", name=f"lot_{taille_lot}"),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
  )
  trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
  exactitude_val = float(trainer.callback_metrics["exactitude_val"])
  print(f"Batchs de {taille_lot} : exactitude_val {exactitude_val:.3f}")

# Epoch par epoch, les petits batchs convergent bien plus vite : 92 % d'exactitude
# de validation dès la première epoch et 95 % à la dixième, quand les batchs de
# 1 500 partent de 51 % et finissent vers 93 %. Chaque epoch voit les mêmes 48 000
# exemples, mais fait 375 mises à jour des poids avec des batchs de 128
# (48 000 / 128), et 32 seulement avec des batchs de 1 500. Le gradient d'un grand
# batch est moins bruité, mais au même learning rate, un pas d'Adam garde à peu près
# la même taille : 12 fois moins de pas, c'est moins de chemin parcouru par epoch.
# Une epoch de grands batchs se calcule en revanche plus vite, et un grand batch
# supporte un learning rate plus grand.

## Recherche d'hyper-paramètres

Combien de neurones dans la couche cachée, quel learning rate ? [Optuna](https://optuna.org/) essaie des combinaisons d'hyper-paramètres et garde la meilleure, notée sur l'ensemble de validation. Son échantillonneur par défaut, TPE, tire ses 10 premiers essais au hasard, puis choisit chaque essai d'après les résultats des précédents (slide « Pour aller plus loin : recherche guidée et élagage »).

Écrivez une fonction objectif, `objectif(essai)`, qui :

- tire les hyper-paramètres de l'essai : `unites`, le nombre de neurones cachés, de 32 à 512 par pas de 32 ; `taux_apprentissage`, le learning rate d'Adam, de 10⁻⁴ à 10⁻², en échelle logarithmique ;
- entraîne un perceptron à une couche cachée ReLU, de sortie `nb_classes` logits, enveloppé dans un `ClassifieurMNIST` avec l'optimiseur `torch.optim.Adam`, sur les labels entiers : 3 epochs, des batchs de 1 500 exemples, 20 % de validation ;
- renvoie son exactitude de validation finale.

Créez ensuite une étude qui maximise cet objectif, lancez-la sur 20 essais, puis affichez le tableau des essais, les meilleurs hyper-paramètres et la meilleure exactitude, et l'historique de la recherche (`plot_optimization_history`). Les essais guidés, après les 10 premiers, font-ils mieux que les tirages au hasard ? Comparez la meilleure exactitude à l'exactitude de validation de `classifieur_profond`, puis comparez les essais entre eux : lequel des deux hyper-paramètres compte le plus, et pourquoi ?

C'est la partie la plus longue : quelques minutes sur CPU.

**Documentation utile**

- [`optuna.trial.Trial`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html)
- [`optuna.create_study`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.create_study.html)
- [`optuna.study.Study.optimize`](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html#optuna.study.Study.optimize)
- [`optuna.visualization.plot_optimization_history`](https://optuna.readthedocs.io/en/stable/reference/visualization/generated/optuna.visualization.plot_optimization_history.html)

In [ ]:
def objectif(essai: optuna.Trial) -> float: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `essai.suggest_int("unites", 32, 512, step=32)` et `essai.suggest_float("taux_apprentissage", 1e-4, 1e-2, log=True)` renvoient la valeur de l'essai en cours
- `nn.Sequential(nn.Linear(dim_entree, unites), nn.ReLU(), nn.Linear(unites, nb_classes))`
- `lightning.Trainer(..., logger=False, enable_progress_bar=False, enable_model_summary=False)` : un entraînement silencieux, pour que les 20 entraînements ne remplissent pas la sortie
- `trainer.callback_metrics["exactitude_val"]` : l'exactitude de validation de la dernière epoch, un tenseur
- `optuna.create_study(direction="maximize")` : une étude qui cherche le maximum de l'objectif ; `etude.optimize(objectif, n_trials=20)` la lance
- `etude.trials_dataframe()`, `etude.best_params` et `etude.best_value` : les essais, les meilleurs hyper-paramètres et la meilleure valeur de l'objectif
- `optuna.visualization.plot_optimization_history(etude).show()` : l'exactitude de chaque essai, et la meilleure obtenue jusque-là
- Pour l'expliquer, comptez les mises à jour des poids de chaque entraînement : les 48 000 exemples d'entraînement divisés par la taille des batchs, fois le nombre d'epochs

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def objectif(essai: optuna.Trial) -> float:
  unites = essai.suggest_int("unites", ...)
  taux_apprentissage = essai.suggest_float("taux_apprentissage", ...)
  modele = nn.Sequential(
    nn.Linear(dim_entree, ...), nn.ReLU(), nn.Linear(..., nb_classes)
  )
  classifieur = ClassifieurMNIST(modele, optimiseur=..., taux_apprentissage=...)
  chargeur_entrainement, chargeur_val = fabrique_chargeurs(
    X_train, y_train, taille_lot=...
  )
  trainer = lightning.Trainer(
    max_epochs=...,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
  )
  trainer.fit(...)
  return float(trainer.callback_metrics[...])


etude = optuna.create_study(direction=...)
etude.optimize(objectif, n_trials=...)
print(etude.trials_dataframe())
print(etude.best_params, etude.best_value)
optuna.visualization.plot_optimization_history(...).show()
```

### Solution

In [ ]:
def objectif(essai: optuna.Trial) -> float:
  unites = essai.suggest_int("unites", 32, 512, step=32)
  taux_apprentissage = essai.suggest_float("taux_apprentissage", 1e-4, 1e-2, log=True)

  modele = nn.Sequential(
    nn.Linear(dim_entree, unites), nn.ReLU(), nn.Linear(unites, nb_classes)
  )
  classifieur = ClassifieurMNIST(
    modele, optimiseur=torch.optim.Adam, taux_apprentissage=taux_apprentissage
  )
  chargeur_entrainement, chargeur_val = fabrique_chargeurs(
    X_train, y_train, taille_lot=1500
  )
  # Un entraînement court, pour que la recherche ait le temps de faire 20 essais
  trainer = lightning.Trainer(
    max_epochs=3,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
  )
  trainer.fit(classifieur, chargeur_entrainement, chargeur_val)
  return float(trainer.callback_metrics["exactitude_val"])


etude = optuna.create_study(direction="maximize", study_name="mnist")

In [ ]:
etude.optimize(objectif, n_trials=20)

In [ ]:
print(
  etude.trials_dataframe()[
    ["number", "value", "params_unites", "params_taux_apprentissage"]
  ]
)
print(f"Meilleurs hyper-paramètres : {etude.best_params}")
print(f"Meilleure exactitude de validation : {etude.best_value:.4f}")
optuna.visualization.plot_optimization_history(etude).show()

Le learning rate compte plus que le nombre de neurones. Dans une exécution, les essais au learning rate proche de 10⁻⁴ restent sous 85 % d'exactitude de validation, quelle que soit leur taille, quand tous ceux au-dessus de 4·10⁻³ atteignent 95 %. Avec des batchs de 1 500 exemples, 3 epochs ne font que 96 mises à jour des poids (48 000 / 1 500 × 3), 12 fois moins par epoch qu'avec des batchs de 128, comme dans la partie TensorBoard : en si peu de pas, seul un grand learning rate converge. La recherche classe donc les configurations pour ce budget d'entraînement, pas pour ce qu'elles donneraient entraînées plus longtemps.

L'historique montre le travail de TPE. Ses 10 premiers essais sont tirés au hasard (le paramètre `n_startup_trials` de `optuna.samplers.TPESampler`), le temps de réunir assez de résultats pour modéliser le score, et s'étalent ici de 69 % à 95 %. Les suivants se concentrent sur les grands learning rates, et tous atteignent 95 %.

Comparé à `classifieur_profond` (environ 95 % d'exactitude de validation, en 3 750 mises à jour des poids), le meilleur essai fait mieux en 96 mises à jour : 97,2 % dans cette exécution (512 neurones, learning rate de 8,3·10⁻³), mais avec 24 fois plus de paramètres (407 050).